# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nithishreddy08/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one pseudonymized content item. The starter dataset contains 30,000 content items across 32 clients, with trailing-90-day performance metrics.

In [17]:
!wget -qO- https://api.github.com/repos/nithishreddy08/flyrank/git/trees/main?recursive=1 | grep "content_refresh"

      "path": "data/raw/content_refresh_anonymized.csv",


In [18]:
import pandas as pd

url = "https://raw.githubusercontent.com/nithishreddy08/flyrank/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())

Rows: 30000
Columns: 44
Clients: 32


In [20]:
!git clone https://github.com/nithishreddy08/flyrank.git
import os
os.chdir("flyrank")  # repo root ki vellu

# File path: data/raw/content_refresh_anonymized.csv
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())

Cloning into 'flyrank'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 163 (delta 69), reused 94 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.86 MiB | 13.82 MiB/s, done.
Resolving deltas: 100% (69/69), done.
Rows: 30000
Columns: 44
Clients: 32


In [24]:
import pandas as pd

url = "https://raw.githubusercontent.com/nithishreddy08/flyrank/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, provider_used, model_used, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, content_age_days, age_tier, age_tier_order, days_since_last_update, freshness_tier, word_count_tier, char_count_tier, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, impression_tier, position_tier

Label / Proxy: trend_direction and trend_pct are label-derived fields and must not be used as features.

Context: content_id and client_id are pseudonymous identifiers used for grouping, joining, and splitting, not as model features.

Excluded: trend_direction and trend_pct are excluded because trend_direction is computed from trend_pct and the declining label is derived from trend_direction. Using them as features would cause target leakage. content_id and client_id are excluded from model features because they are identifiers rather than meaningful predictive variables.

In [26]:
for i, col in enumerate(df.columns, 1):
    print(i, col)

1 content_id
2 client_id
3 search_volume
4 competition
5 competition_level
6 cpc
7 content_type
8 main_intent
9 word_count
10 char_count
11 provider_used
12 model_used
13 impressions_90d
14 clicks_90d
15 pageviews_90d
16 sessions_90d
17 users_90d
18 engaged_sessions_90d
19 ai_sessions_90d
20 scroll_events_90d
21 days_with_impressions
22 days_with_sessions
23 impressions_last_30d
24 clicks_last_30d
25 sessions_last_30d
26 impressions_prev_30d
27 clicks_prev_30d
28 sessions_prev_30d
29 content_age_days
30 age_tier
31 age_tier_order
32 days_since_last_update
33 freshness_tier
34 word_count_tier
35 char_count_tier
36 ctr
37 avg_position
38 engagement_rate
39 scroll_rate
40 ai_traffic_pct
41 impression_tier
42 position_tier
43 trend_direction
44 trend_pct


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain verification: Each content_id appears only once. The dataset has 30,000 unique content_id values and 0 duplicate content IDs, confirming that one row represents one pseudonymized content item. There are 32 unique clients. Missingness verification: Missing values are not distributed randomly across content types. For example, all 2,096 feedly article rows have missing search_volume, while keyword articles contain most of the missing word_count and provider_used values. This confirms that missingness follows content_type and should not be blindly replaced with zero. Time-window verification: The starter dataset contains trailing-90-day metrics rather than a single row-level calendar date. Therefore, the analysis treats the performance metrics as trailing 90-day measures and does not claim a specific calendar start or end date without a date field.

In [27]:
print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())
print("Unique clients:", df["client_id"].nunique())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))

Total rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0
Unique clients: 32

Missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
dtype: int64


In [28]:
missing_by_type = df.groupby("content_type").agg(
    rows=("content_id", "size"),
    missing_word_count=("word_count", lambda x: x.isna().sum()),
    missing_provider=("provider_used", lambda x: x.isna().sum()),
    missing_search_volume=("search_volume", lambda x: x.isna().sum())
)

missing_by_type

,rows,missing_word_count,missing_provider,missing_search_volume
content_type,,,,
comparison article,697,0,581,0
feedly article,2096,0,1468,2096
keyword article,27207,7699,19389,372


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits:

The starter dataset contains trailing-90-day metrics, so it does not provide a full daily history of performance.

Missingness is patterned by content_type. For example, search_volume is missing for all feedly article rows, and word_count is missing for a substantial portion of keyword articles. Therefore, missing values cannot automatically be interpreted as zero.

content_id and client_id are pseudonymous identifiers. They can be used for grouping, joining, and splitting, but they do not provide meaningful predictive information.

trend_direction and trend_pct should not be used as model features because they are directly related to the outcome/label and can cause target leakage.

The dataset represents 32 clients, so results may not generalize to clients or content types that are not represented in this sample.

In [30]:
print("Total rows:", len(df))
print("Total clients:", df["client_id"].nunique())
print("\nContent types:")
print(df["content_type"].value_counts())

Total rows: 30000
Total clients: 32

Content types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.